In [23]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import sys
sys.path.append("../")

import nest_asyncio
nest_asyncio.apply()

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "vscode"            

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import matplotlib.dates as mdates
plt.style.use('seaborn-v0_8-dark')
params = {'legend.fontsize': 'x-large',
        'figure.figsize': (16, 9),
        'axes.labelsize': 'x-large',
        'axes.titlesize':'x-large',
        'xtick.labelsize':'x-large',
        'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np
import QuantLib as ql
import rateslib as rl

import datetime
import pytz
NY_tz = pytz.timezone("America/New_York") 
CHI_tz = pytz.timezone("America/Chicago") 
UTC_tz = pytz.timezone("UTC")

import warnings
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [24]:
# cms_spread_options_single_look = [
#     "QZQHDJR8ZR2C",
# 	"QZSP0NTLKKFJ",
# 	"QZP43JLWTM5W",
# 	"QZVH3T5N6GJ9",
# 	"QZD8FJ9CGMZ2",
# 	"QZH64P6BDDFN",
# 	"QZKC9F9FV3ZV",
# ]
# df[df["UPI Underlier Name"] == "USD-SOFR ICE Swap Rate vs USD-SOFR-COMPOUND"].to_csv("USD-SOFR ICE Swap Rate vs USD-SOFR-COMPOUND sdr trades.csv")

In [25]:
from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP
from SDRUtils.products.usd.sofr_swaps import USD_SOFR_SwapProduct 
from SDRUtils.products.usd.usd_swaptions import USD_Swaptions

In [26]:
cache_path = r"C:\Users\chris\clee\project-oasis\private\sdranalytics\.cache"

# as_of = datetime.date(2026, 2, 23)
# start = NY_tz.localize(datetime.datetime(as_of.year, as_of.month, as_of.day, 0, 0))
# end = NY_tz.localize(datetime.datetime(as_of.year, as_of.month, as_of.day, 23, 59))

start = NY_tz.localize(datetime.datetime(2026, 3, 9, 0, 0))
end = NY_tz.localize(datetime.datetime(2026, 3, 9, 23, 59))

# mdp = IRSwapsMDP(source="ERIS_EOD_LIVE-QL_BASIC")
# pricer = mdp.get_pricer(request=dict(curve_name="USD-SOFR-1D", timestamp=start.date()))

from SDRUtils.data.builder import SDRDataBuilder
sdr = SDRDataBuilder(cache_path=cache_path, show_tqdm=True)
df = sdr.grab_sdr_trades(
	start_timestamp=start,
	end_timestamp=end,
	agency="CFTC",
	asset_class="RATES",
)
df

FETCHING INTRADAY SDR SLICES...: 100%|██████████| 10/10 [00:00<00:00, 12.81it/s]


,Dissemination Identifier,Original Dissemination Identifier,Action type,Event type,Event timestamp,Amendment indicator,Asset Class,Product name,Cleared,Mandatory clearing indicator,...,Package transaction price currency,Package transaction price notation,Package transaction spread,Package transaction spread currency,Package transaction spread notation,Physical delivery location-Leg 1,Delivery Type,Unique Product Identifier,UPI FISN,UPI Underlier Name
0,2293643764000000101,<NA>,NEWT,TRAD,2026-03-09 04:00:05+00:00,NaN,IR,NaN,I,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,QZXQ4R16245X,NA/Swap OIS USD,USD-SOFR-COMPOUND
1,2293643980000000101,<NA>,NEWT,TRAD,2026-03-09 04:00:23+00:00,NaN,IR,NaN,I,True,...,NaN,NaN,0.3503,NaN,3.0,NaN,NaN,QZF08M5TR8H3,NA/Swap OIS JPY,JPY-TONA-OIS Compound
2,2293643982000000301,1947848289000001201,MODI,TRAD,2026-03-09 04:00:29+00:00,True,IR,NaN,N,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,QZ8QCCTNW2LL,NA/O Nstd Oth EUR,EUR-EURIBOR ICE Swap Rate-11:00
3,2293644825000000101,<NA>,NEWT,TRAD,2026-03-09 04:00:37+00:00,False,IR,NaN,I,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,QZJQHSR2XNCJ,NA/Swap OIS SGD,SGD-SORA-OIS Compound
4,2293708034000001101,<NA>,NEWT,TRAD,2026-03-09 04:00:38+00:00,NaN,IR,NaN,I,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,QZVNRZSP6KHF,NA/Swap Fxd Flt INR,INR-MIBOR-OIS-COMPOUND
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12515,2304570885000001401,,NEWT,TRAD,2026-03-09 19:24:34+00:00,None,IR,None,I,True,...,,NaN,NaN,None,NaN,None,None,QZXQ4R16245X,NA/Swap OIS USD,USD-SOFR-COMPOUND
12516,2304572090000001101,,NEWT,TRAD,2026-03-09 19:25:08+00:00,None,IR,None,I,True,...,,NaN,-0.004437,None,3.0,None,None,QZPB5VSBGRCD,NA/Swap OIS USD,USD-SOFR-OIS Compound
12517,2304572083000000401,,NEWT,TRAD,2026-03-09 19:25:18+00:00,None,IR,None,I,False,...,USD,1.0,NaN,None,NaN,None,None,QZXQ4R16245X,NA/Swap OIS USD,USD-SOFR-COMPOUND
12518,2304572141000000401,,NEWT,TRAD,2026-03-09 19:25:26+00:00,None,IR,None,I,False,...,,NaN,NaN,None,NaN,None,None,QZJB92X0X668,NA/Swap OIS MXN,MXN-TIIE ON-OIS Compound


In [28]:
# df[df["Notional currency-Leg 1"] == "USD"]
capfloor_upifisn = [
    "NA/O Call Epn USD",
    "NA/O P Epn USD",
]

s = df["Option Premium Amount"].astype("string").str.strip()
s = (s
     .str.replace(r"[\$,]", "", regex=True)                 
     .str.replace(r"^\((.*)\)$", r"-\1", regex=True)        
)
df["Option Premium Amount"] = pd.to_numeric(s, errors="coerce")

out = df[
    df["UPI FISN"].isin(capfloor_upifisn)
    & (df["Package indicator"] == False)
    & (df["Action type"] == "NEWT")
    # & (df["Option Premium Amount"] > 0)
    & (df["UPI Underlier Name"].str.contains("SOFR"))
]
out
# out.to_csv("usd_capfloor_sdr_trades_last_week_feb2026.csv")
# ["Unique Product Identifier"].value_counts()

# out["Unique Product Identifier"].value_counts()
# out["Platform identifier"].value_counts()

,Dissemination Identifier,Original Dissemination Identifier,Action type,Event type,Event timestamp,Amendment indicator,Asset Class,Product name,Cleared,Mandatory clearing indicator,...,Package transaction price currency,Package transaction price notation,Package transaction spread,Package transaction spread currency,Package transaction spread notation,Physical delivery location-Leg 1,Delivery Type,Unique Product Identifier,UPI FISN,UPI Underlier Name
7315,2300920322000001101,<NA>,NEWT,TRAD,2026-03-09 12:54:22+00:00,NaN,IR,NaN,N,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,QZXNP136XML0,NA/O Call Epn USD,USD-SOFR CME Term
9577,2302066140000000901,<NA>,NEWT,TRAD,2026-03-09 14:42:26+00:00,NaN,IR,NaN,N,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,QZQJWDQ4V0VJ,NA/O Call Epn USD,USD-SOFR CME Term
10674,2302366154000000201,<NA>,NEWT,TRAD,2026-03-09 15:54:31+00:00,NaN,IR,NaN,N,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,QZQJWDQ4V0VJ,NA/O Call Epn USD,USD-SOFR CME Term
11257,2302463155000000601,<NA>,NEWT,TRAD,2026-03-09 16:37:49+00:00,NaN,IR,NaN,N,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,QZXNP136XML0,NA/O Call Epn USD,USD-SOFR CME Term
12008,2303567577000000301,<NA>,NEWT,TRAD,2026-03-09 17:50:32+00:00,NaN,IR,NaN,N,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,QZQJWDQ4V0VJ,NA/O Call Epn USD,USD-SOFR CME Term
12494,2304572138000000101,,NEWT,TRAD,2026-03-09 19:22:15+00:00,None,IR,None,N,False,...,,NaN,NaN,None,NaN,None,None,QZQJWDQ4V0VJ,NA/O Call Epn USD,USD-SOFR CME Term


In [20]:
import numpy as np
from scipy.optimize import brentq
from scipy.stats import norm
from dateutil.relativedelta import relativedelta


# ── Bachelier caplet pricer ──────────────────────────────────────────────────

def bachelier_caplet(F, K, sigma, T, tau, DF, N):
    """
    Price a single caplet under the Bachelier (normal) model.

    Parameters
    ----------
    F     : forward rate (decimal, e.g. 0.0430)
    K     : strike (decimal)
    sigma : normal vol (annualised, in rate terms e.g. 0.0050 = 50bp)
    T     : time to fixing in years (ACT/365F for vol scaling)
    tau   : accrual fraction for the caplet period (ACT/360)
    DF    : discount factor to payment date
    N     : notional

    Returns
    -------
    float : caplet premium in currency units
    """
    if T < 1e-10:
        # caplet is at or past fixing — intrinsic only
        return N * tau * DF * max(F - K, 0.0)

    sqrt_T = np.sqrt(T)
    d = (F - K) / (sigma * sqrt_T)
    price = N * tau * DF * (sigma * sqrt_T * norm.pdf(d) + (F - K) * norm.cdf(d))
    return price


# ── Schedule builder ─────────────────────────────────────────────────────────

def build_caplet_schedule(effective, expiration, freq_months=1):
    """
    Generate caplet periods from effective to expiration.

    Returns list of (accrual_start, accrual_end) datetime pairs.
    Each caplet fixes at accrual_start (Term SOFR = forward-looking).
    Payment at accrual_end.
    """
    periods = []
    dt_start = effective
    while dt_start < expiration:
        dt_end = dt_start + relativedelta(months=freq_months)
        if dt_end > expiration:
            dt_end = expiration
        periods.append((dt_start, dt_end))
        dt_start = dt_end
    return periods


# ── Core vol stripper ────────────────────────────────────────────────────────

def strip_cap_vol(trade: dict, curve, valuation_date: datetime = None):
    """
    Back out the flat implied Bachelier vol from a cap/floor SDR trade.

    Parameters
    ----------
    trade : dict
        SDR trade record in DTCC schema (same keys as raw data).
    curve : rateslib.curves.Curve
        Calibrated SOFR OIS curve. Used for both forwards and discounting.
    valuation_date : datetime, optional
        Pricing date. Defaults to today if not provided.

    Returns
    -------
    dict with:
        implied_vol_bps  : flat normal vol in basis points
        implied_vol_dec  : flat normal vol in decimal
        model_premium    : repriced cap premium at solved vol
        market_premium   : observed premium from SDR
        caplet_details   : list of per-caplet diagnostics
    """
    # ── parse trade fields ───────────────────────────────────────────────
    effective = trade["Effective Date"]
    expiration = trade["Expiration Date"]

    if isinstance(effective, str):
        effective = datetime.datetime.strptime(effective, "%Y-%m-%d")
    elif hasattr(effective, "to_pydatetime"):
        effective = effective.to_pydatetime().replace(tzinfo=None)

    if isinstance(expiration, str):
        expiration = datetime.datetime.strptime(expiration, "%Y-%m-%d")
    elif hasattr(expiration, "to_pydatetime"):
        expiration = expiration.to_pydatetime().replace(tzinfo=None)

    notional_str = str(trade["Notional amount-Leg 1"]).replace(",", "")
    N = float(notional_str)

    # strike: notation 3.0 = percentage points
    K = float(trade["Strike Price"])
    if trade.get("Strike price notation") == 3.0:
        # already in decimal if <= 1, percentage if > 1
        if K > 1:
            K = K / 100.0  # e.g. 4.5 -> 0.045
        # if already 0.045 from SDR, leave as-is

    premium = float(trade["Option Premium Amount"])

    freq_months = int(trade.get("Floating rate reset frequency period multiplier-leg 1", 1))

    if valuation_date is None:
        valuation_date = datetime.datetime.today().replace(hour=0, minute=0, second=0, microsecond=0)

    # ── build caplet schedule ────────────────────────────────────────────
    periods = build_caplet_schedule(effective, expiration, freq_months)

    # ── extract forwards and DFs from rateslib curve ─────────────────────
    caplets = []
    for accrual_start, accrual_end in periods:
        # forward rate for the period (rateslib returns in percentage — depends on convention)
        fwd_rate = float(curve.rate(accrual_start, accrual_end))
        # rateslib .rate() returns in percentage terms (e.g. 4.30 for 4.30%)
        F = fwd_rate / 100.0

        # discount factor to payment date
        # DF(t) = curve[accrual_end] / curve[valuation_date], but since
        # curve initial node = 1.0 at its own start date, we just need:
        df_pay = float(curve[accrual_end])

        # accrual fraction ACT/360
        day_count = (accrual_end - accrual_start).days
        tau = day_count / 360.0

        # time to fixing in years (ACT/365F for vol time-scaling)
        T = max((accrual_start - valuation_date).days / 365.0, 0.0)

        caplets.append({
            "accrual_start": accrual_start,
            "accrual_end": accrual_end,
            "forward": F,
            "discount_factor": df_pay,
            "tau": tau,
            "T_fix": T,
            "days_to_fix": (accrual_start - valuation_date).days,
        })

    # ── objective: sum of caplet prices = market premium ──────────────────
    def price_cap(sigma):
        return sum(
            bachelier_caplet(c["forward"], K, sigma, c["T_fix"], c["tau"], c["discount_factor"], N)
            for c in caplets
        )

    def objective(sigma):
        return price_cap(sigma) - premium

    # ── solve ─────────────────────────────────────────────────────────────
    # search over 0.1bp to 500bp normal vol
    sigma_lo = 0.000001  # ~0.01bp
    sigma_hi = 0.0500    # 500bp

    # check bounds
    p_lo = objective(sigma_lo)
    p_hi = objective(sigma_hi)

    if p_lo * p_hi > 0:
        # check if premium is below intrinsic
        intrinsic = price_cap(0.0)
        if premium <= intrinsic + 1e-2:
            raise ValueError(
                f"Premium ${premium:,.0f} is at or below intrinsic ${intrinsic:,.2f}. "
                f"No vol to extract — cap is deep ITM or premium is misreported."
            )
        raise ValueError(
            f"Cannot bracket root. Cap price at {sigma_hi*1e4:.0f}bp vol = "
            f"${price_cap(sigma_hi):,.0f} vs market ${premium:,.0f}. "
            f"Check inputs."
        )

    implied_sigma = brentq(objective, sigma_lo, sigma_hi, xtol=1e-12, maxiter=200)

    # ── per-caplet diagnostics at solved vol ──────────────────────────────
    for c in caplets:
        c["caplet_price"] = bachelier_caplet(
            c["forward"], K, implied_sigma, c["T_fix"], c["tau"], c["discount_factor"], N
        )
        c["moneyness_bps"] = (c["forward"] - K) * 10000

    return {
        "implied_vol_bps": implied_sigma * 10000,
        "implied_vol_dec": implied_sigma,
        "model_premium": price_cap(implied_sigma),
        "market_premium": premium,
        "strike": K,
        "notional": N,
        "num_caplets": len(caplets),
        "caplet_details": caplets,
    }

In [21]:
from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP
curve_mdp = IRSwapsMDP(source="BARCHART_STIRF-RL")
curve_handle = curve_mdp._get_curve(curve_name="USD-SOFR-1D-Q12xM12STIRT", timestamp=NY_tz.localize(datetime.datetime(2026, 2, 27, 17, 00)))
# curve_handle.handle().plot("1d")

In [22]:

trade = df[df["Unique Product Identifier"] == "QZQJWDQ4V0VJ"].iloc[0].to_dict()

result = strip_cap_vol(trade, curve_handle.handle(), valuation_date=datetime.datetime(2026, 2, 27))

print(f"Implied Normal Vol: {result['implied_vol_bps']:.2f} bps")
print(f"Model Premium:      ${result['model_premium']:,.2f}")
print(f"Market Premium:     ${result['market_premium']:,.2f}")
print()
print(f"{'Period':<6} {'Start':<12} {'End':<12} {'Fwd':>7} {'Moneyness':>10} {'T_fix':>7} {'Price':>10}")
print("-" * 75)
for i, c in enumerate(result["caplet_details"]):
	print(
		f"{i+1:<6} {c['accrual_start'].strftime('%Y-%m-%d'):<12} "
		f"{c['accrual_end'].strftime('%Y-%m-%d'):<12} "
		f"{c['forward']*100:>6.3f}% "
		f"{c['moneyness_bps']:>+8.1f}bp "
		f"{c['T_fix']:>6.3f}y "
		f"${c['caplet_price']:>9,.2f}"
	)

Implied Normal Vol: 82.81 bps
Model Premium:      $3,700.00
Market Premium:     $3,700.00

Period Start        End              Fwd  Moneyness   T_fix      Price
---------------------------------------------------------------------------
1      2026-03-01   2026-04-01    3.684%    -81.6bp  0.005y $     0.00
2      2026-04-01   2026-05-01    3.655%    -84.5bp  0.090y $     1.44
3      2026-05-01   2026-06-01    3.603%    -89.7bp  0.173y $    32.68
4      2026-06-01   2026-07-01    3.563%    -93.7bp  0.258y $   121.36
5      2026-07-01   2026-08-01    3.508%    -99.2bp  0.340y $   233.83
6      2026-08-01   2026-09-01    3.413%   -108.7bp  0.425y $   290.33
7      2026-09-01   2026-10-01    3.355%   -114.5bp  0.510y $   378.15
8      2026-10-01   2026-11-01    3.284%   -121.6bp  0.592y $   452.15
9      2026-11-01   2026-12-01    3.191%   -130.9bp  0.677y $   451.30
10     2026-12-01   2027-01-01    3.125%   -137.5bp  0.759y $   511.56
11     2027-01-01   2027-02-01    3.091%   -140.9bp 

In [15]:
df[df["Unique Product Identifier"] == "QZQJWDQ4V0VJ"].iloc[0].to_dict()

{'Dissemination Identifier': '2195041583000000201',
 'Original Dissemination Identifier': '',
 'Action type': 'NEWT',
 'Event type': 'TRAD',
 'Event timestamp': Timestamp('2026-02-27 14:45:43+0000', tz='UTC'),
 'Amendment indicator': None,
 'Asset Class': 'IR',
 'Product name': None,
 'Cleared': 'N',
 'Mandatory clearing indicator': False,
 'Execution Timestamp': Timestamp('2026-02-27 14:45:43+0000', tz='UTC'),
 'Effective Date': Timestamp('2026-03-01 00:00:00'),
 'Expiration Date': Timestamp('2027-03-01 00:00:00'),
 'Maturity date of the underlier': None,
 'Non-standardized term indicator': True,
 'Platform identifier': 'BILT',
 'Prime brokerage transaction indicator': False,
 'Block trade election indicator': False,
 'Large notional off-facility swap election indicator': False,
 'Notional amount-Leg 1': '78,000,000',
 'Notional amount-Leg 2': '',
 'Notional currency-Leg 1': 'USD',
 'Notional currency-Leg 2': '',
 'Notional quantity-Leg 1': None,
 'Notional quantity-Leg 2': None,
 'To

In [5]:
df[
    # (df["UPI Underlier Name"].str.lower().str.contains("vs"))
    # & (df["UPI Underlier Name"].str.lower().str.contains("usd"))
    # & (df["UPI Underlier Name"].str.lower().str.contains("cad"))
    (df["UPI Underlier Name"].str.lower().str.contains("gbp"))
]["UPI Underlier Name"].value_counts()
# .to_csv("02-26-2026-usdcad-xccy-basis-sdr.csv")
# ["UPI Underlier Name"].value_counts()

UPI Underlier Name
GBP-SONIA-COMPOUND                                 1048
GBP-SONIA-OIS Compound                              979
GBP-SONIA-OIS Compound vs USD-SOFR-OIS Compound     160
NA/Swap OIS GBP                                     152
NA/Swap Fxd Flt GBP                                  19
GBP-SONIA-COMPOUND vs USD-SOFR-COMPOUND               7
GBP-WMBA-SONIA-COMPOUND                               5
GBP-SONIA                                             4
GBP-SONIA vs USD-SOFR                                 3
GBP-LIBOR-BBA vs USD-SOFR                             2
GBP-SONIA-OIS Compound vs OTHER                       1
Name: count, dtype: int64

In [6]:
# start = NY_tz.localize(datetime.datetime(2026, 3, 26, 0, 0))
# end = NY_tz.localize(datetime.datetime(2026, 2, 26, 23, 59))

sdf = USD_Swaptions().build_classification_dataframe(start=start, end=end, cache_path=cache_path, merge_package_legs=False)
# sdf = USD_SOFR_SwapProduct().build_classification_dataframe(start=start, end=end, cache_path=cache_path, ignore_cache=False, merge_package_legs=False)

PRICING STRADDLES...: 100%|██████████| 2/2 [00:00<00:00, 246.05it/s]


SUCCESS: `func_tol` reached after 0 iterations (levenberg_marquardt), `f_val`: 1.5467787414347294e-26, `time`: 0.0028s
SUCCESS: `func_tol` reached after 0 iterations (levenberg_marquardt), `f_val`: 1.5469759566610347e-26, `time`: 0.0039s


PRICING OUTRIGHTS...: 100%|██████████| 47/47 [00:00<00:00, 177.19it/s]


In [8]:
sdf.tail(4)

,event_action,trade_id,execution_timestamp,effective_date,expiration_date,product_type,trade_label,notional,notional_currency,is_notional_capped,...,outright_vega01,outright_gamma01,outright_theta1d,matched_ust_maturity,matched_ust_maturity_trade_confidence,ust_cusip,ust_oi,ust_issue_date,swap_maturity_date,invoice_swap_ticker
45,NEWT-TRAD,2215165994000000101,2026-03-02 17:08:14+00:00,2026-03-02,2031-02-10,SWAPTION_PAYER,USD-SOFR-COMPOUND 1D Constant 4Y11MxIMM_H2031 ...,15000000.0,USD,False,...,NaN,NaN,NaN,False,<NA>,<NA>,NaN,NaN,2031-03-17,None
46,NEWT-TRAD,2215232564000000201,2026-03-02 17:12:43+00:00,2026-03-02,2030-03-04,SWAPTION_PAYER,USD-SOFR-COMPOUND 1D Constant 4Yx1Y PAYER BERM...,5000000.0,USD,False,...,NaN,NaN,NaN,False,<NA>,<NA>,NaN,NaN,2031-03-10,None
47,NEWT-TRAD,2215263471000000601,2026-03-02 17:13:59+00:00,2026-03-02,2030-03-04,SWAPTION_PAYER,USD-SOFR-COMPOUND 1D Constant 4Yx1Y PAYER BERM...,5000000.0,USD,False,...,NaN,NaN,NaN,False,<NA>,<NA>,NaN,NaN,2031-03-10,None
48,NEWT-TRAD,2215299715000000501,2026-03-02 17:16:27+00:00,2026-03-02,2030-12-04,SWAPTION_PAYER,USD-SOFR-COMPOUND 1D Constant 4Y9Mx3M PAYER BE...,5000000.0,USD,False,...,NaN,NaN,NaN,False,<NA>,<NA>,NaN,NaN,2031-03-11,None
